<a href="https://colab.research.google.com/github/Ahmadlazz/data_acquisition/blob/main/Scrapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import requests
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

TARGET = "https://indonesiakaya.com/"
UA = "INSTIKI-BIFP243-Praktikum/1.0 (mahasiswa@instiki.ac.id)"

def cek_kepatuhan(url, user_agent=UA):
    p = urlparse(url)
    base = f"{p.scheme}://{p.netloc}"
    rp = RobotFileParser(); rp.set_url(base + "/robots.txt")
    try:
        rp.read(); isi = requests.get(base + "/robots.txt", timeout=10).text
    except Exception as e:
        return {"error": str(e)}
    return {
        "url": url,
        "boleh_diakses": rp.can_fetch(user_agent, url),
        "crawl_delay": rp.crawl_delay(user_agent),
        "isi_robots_txt": isi[:300],
    }

hasil = cek_kepatuhan(TARGET)
for k, v in hasil.items():
    print(f"{k}: {v}\n")

url: https://indonesiakaya.com/

boleh_diakses: False

crawl_delay: None

isi_robots_txt: # START YOAST BLOCK
# ---------------------------
User-agent: *
Disallow:

Sitemap: https://indonesiakaya.com/sitemap_index.xml
# ---------------------------
# END YOAST BLOCK



In [3]:
from bs4 import BeautifulSoup
r = requests.get(TARGET, headers={"User-Agent": UA}, timeout=10)
r.raise_for_status()
soup = BeautifulSoup(r.text, "lxml")
print("Judul halaman:", soup.title.string.strip())
print("Jumlah tag <div>:", len(soup.find_all("div")))
print("\nCuplikan struktur satu kutipan:")
print(soup.select_one("div.quote").prettify()[:500])

Judul halaman: Quotes to Scrape
Jumlah tag <div>: 28

Cuplikan struktur satu kutipan:
<div class="quote" itemscope="" itemtype="http://schema.org/CreativeWork">
 <span class="text" itemprop="text">
  “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
 </span>
 <span>
  by
  <small class="author" itemprop="author">
   Albert Einstein
  </small>
  <a href="/author/Albert-Einstein">
   (about)
  </a>
 </span>
 <div class="tags">
  Tags:
  <meta class="keywords" content="change,deep-thoughts,thinking,world" itemprop="ke


In [5]:
baris = []
for q in soup.select("div.quote"):
 baris.append({
 "teks": q.select_one("span.text").get_text(strip=True),
 "penulis": q.select_one("small.author").get_text(strip=True),
 "tag": "|".join(t.get_text(strip=True) for t in q.select("div.tagsa.tag")),
 "url_penulis": q.select_one("span a")["href"],
 })
import pandas as pd
df = pd.DataFrame(baris)
print("Jumlah kutipan di halaman 1:", len(df))
print(df.head())

Jumlah kutipan di halaman 1: 10
                                                teks          penulis tag  \
0  “The world as we have created it is a process ...  Albert Einstein       
1  “It is our choices, Harry, that show what we t...     J.K. Rowling       
2  “There are only two ways to live your life. On...  Albert Einstein       
3  “The person, be it gentleman or lady, who has ...      Jane Austen       
4  “Imperfection is beauty, madness is genius and...   Marilyn Monroe       

               url_penulis  
0  /author/Albert-Einstein  
1      /author/J-K-Rowling  
2  /author/Albert-Einstein  
3      /author/Jane-Austen  
4   /author/Marilyn-Monroe  


In [17]:
import time
from urllib.parse import urljoin

def scrape_kutipan(base="https://indonesiakaya.com/", maks_halaman=5, jeda=1.5):
    semua, url = [], base
    for halaman in range(1, maks_halaman + 1):
        r = requests.get(url, headers={"User-Agent": UA}, timeout=10)
        if r.status_code != 200:
            print("Berhenti, status:", r.status_code)
            break
        s = BeautifulSoup(r.text, "lxml")
        kutipan = s.select("div.quote")
        if not kutipan:
            print("Tidak ada data lagi.")
            break
        for q in kutipan:
            semua.append({
                "teks": q.select_one("span.text").get_text(strip=True),
                "penulis": q.select_one("small.author").get_text(strip=True),
                "tag": "|".join(t.get_text(strip=True) for t in q.select("a.tag")),
                "halaman": halaman,
                "url_sumber": url,
            })
        print(f"halaman {halaman}: {len(kutipan)} item")
        nxt = s.select_one("li.next a")
        if not nxt:
            print("Halaman terakhir tercapai.")
            break
        url = urljoin(base, nxt["href"])
        time.sleep(jeda)  # jeda sopan - JANGAN dihapus
    return pd.DataFrame(semua)

kutipan_df = scrape_kutipan()
print("\nTotal:", kutipan_df.shape)

Tidak ada data lagi.

Total: (0, 0)


In [12]:
html_tabel = """
<table id="kunjungan">
 <thead><tr><th>Bulan</th><th>Domestik</th><th>Mancanegara</th></tr></thead>
 <tbody>
 <tr><td>Januari</td><td>12.000</td><td>8.500</td></tr>
 <tr><td>Februari</td><td>14.500</td><td>9.200</td></tr>
 <tr><td>Maret</td><td>13.100</td><td>10.400</td></tr>
 </tbody>
</table>"""
# Cara cepat
tabel = pd.read_html(html_tabel, thousands=".")[0]
print(tabel)
print(tabel.dtypes)
# Cara manual (untuk struktur tidak standar)
s = BeautifulSoup(html_tabel, "lxml")
header = [th.get_text(strip=True) for th in s.select("thead th")]
rows = [[td.get_text(strip=True) for td in tr.select("td")] for tr in
s.select("tbody tr")]
print("\nManual:", header, rows[0])

      Bulan  Domestik  Mancanegara
0   Januari     12000         8500
1  Februari     14500         9200
2     Maret     13100        10400
Bulan          object
Domestik        int64
Mancanegara     int64
dtype: object

Manual: ['Bulan', 'Domestik', 'Mancanegara'] ['Januari', '12.000', '8.500']


/tmp/ipykernel_733/2500487148.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tabel = pd.read_html(html_tabel, thousands=".")[0]


In [18]:
from datetime import datetime
from pathlib import Path
import json

Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("docs").mkdir(exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_csv = f"data/raw/kutipan_{stamp}.csv"
kutipan_df.to_csv(out_csv, index=False)

meta = {
    "berkas": out_csv,
    "situs_sumber": TARGET,
    "waktu_scraping": datetime.now().isoformat(timespec="seconds"),
    "user_agent": UA,
    "jeda_antar_request_detik": 1.5,
}

with open(f"docs/metadata_{stamp}.json", "w") as f:
    json.dump(meta, f, indent=4)

print("Data dan metadata berhasil disimpan!")

Data dan metadata berhasil disimpan!
